# Solutions · 14 · Chemical reaction equilibrium

Worked solutions to the exercises of [notebook 14](../notebooks/14_chemical_reaction_equilibrium.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engthermo import reaction
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Methanol synthesis (CO + 2 H2 -> CH3OH) is run at 250 degC and 50-100 bar. Compute the equilibrium methanol mole fraction for a stoichiometric feed at 50 and 100 bar, and at 300 degC. Why is the process run with a large recycle?

In [2]:
meoh = {"CO": -1, "H2": -2, "methanol": 1}
for T_C, p_bar in ((250, 50), (250, 100), (300, 100)):
    ex = reaction.extent(meoh, {"CO": 1.0, "H2": 2.0}, T_C + 273.15, p_bar * 1e5)
    print(f"{T_C} degC, {p_bar:3d} bar: y_methanol = {ex['y']['methanol']:.3f}, CO conversion {ex['extent']:.1%}")

250 degC,  50 bar: y_methanol = 0.274, CO conversion 53.1%
250 degC, 100 bar: y_methanol = 0.457, CO conversion 71.7%
300 degC, 100 bar: y_methanol = 0.202, CO conversion 43.1%


The equilibrium conversion is 50-70 % per pass at 250 degC and falls to about 40 % at 300 degC (the reaction
is exothermic), and real reactors stay well below equilibrium because the catalyst is slow at low
temperature. Methanol plants therefore condense the product and recycle the unconverted gas; the per-pass
equilibrium sets the recycle ratio and with it the compressor and reactor sizes.

## Exercise 2

Use Gibbs minimisation to find the equilibrium of the reformer at a steam-to-carbon ratio of 2 and of 4 (900 degC, 20 bar). How does the ratio affect methane conversion and the CO/CO2 split? (Carbon formation is ignored here.)

In [3]:
species = ["CH4", "H2O", "CO", "CO2", "H2"]
for sc in (2.0, 3.0, 4.0):
    gm = reaction.gibbs_minimization(species, {"CH4": 1.0, "H2O": sc}, 900 + 273.15, 20e5)
    n = gm["moles"]
    print(f"steam/carbon {sc:.0f}: CH4 conversion {1 - n['CH4']:.3f}, H2 {n['H2']:.2f} mol, CO/CO2 = {n['CO']/n['CO2']:.2f}")

steam/carbon 2: CH4 conversion 0.839, H2 2.70 mol, CO/CO2 = 3.60
steam/carbon 3: CH4 conversion 0.921, H2 3.05 mol, CO/CO2 = 2.22
steam/carbon 4: CH4 conversion 0.958, H2 3.24 mol, CO/CO2 = 1.59


More steam pushes both the reforming and the shift reactions to the right: higher methane conversion, more
hydrogen and less CO. It also keeps carbon (coke) from forming on the catalyst - the real reason for
steam-to-carbon ratios of about 3 - at the cost of heating and later condensing the extra steam.

## Exercise 3

**Implement it yourself:** solve the ammonia equilibrium extent at 450 degC and 200 bar by writing the equation $K = y_{NH_3}^2/(y_{N_2}y_{H_2}^3)\,(p/p^\circ)^{-2}$ in terms of the extent and using `brentq`; compare with `reaction.extent`.

In [4]:
rxn = {"N2": -1, "H2": -3, "NH3": 2}
T, p = 723.15, 200e5
K = reaction.equilibrium_constant(rxn, T)
def f(xi):                                  # feed 1 N2 + 3 H2; extent xi
    n_N2, n_H2, n_NH3 = 1 - xi, 3 - 3 * xi, 2 * xi
    n = n_N2 + n_H2 + n_NH3
    y = np.array([n_N2, n_H2, n_NH3]) / n
    return y[2]**2 / (y[0] * y[1]**3) * (p / 1e5)**-2 - K
xi = optimize.brentq(f, 1e-9, 1 - 1e-9)
print(f"by hand: extent {xi:.4f}, y_NH3 = {2*xi/(4 - 2*xi):.4f};  library: {reaction.extent(rxn, {'N2': 1.0, 'H2': 3.0}, T, p)['extent']:.4f}")

by hand: extent 0.4005, y_NH3 = 0.2504;  library: 0.4005


Writing the mole fractions in terms of the extent turns the equilibrium condition into one equation in one
unknown on the interval (0, 1). The library solves the same equation in logarithmic form, which is better
conditioned when K is very large or very small.